## Disclaimer
This project is for educational purposes only and does not provide financial advice.

# Financial Risk Assessment Using NLP
This is a highly comprehensive project that involves multiple facets of NLP, including sentiment analysis, named entity recognition, and text classification. This code provides an entire pipeline to assess the financial risk associated with a company, using textual data from news sources. The code uses the following pre-trained models:
1. HuggingFace Sentiment Analysis Model: This model is based on a transformer architecture, typically fine-tuned on datasets like IMDB reviews or Twitter data. It is used to analyze the sentiment of news articles and classify them into positive or negative.  
2. SpaCy's Named Entity Recognition: The en_core_web_sm model is pre-trained on a large corpus of web pages and news data to extract named entities, such as people, organizations, and places, that are relevant for understanding news context.  
The risk score is derived from sentiment analysis and entity recognition, helping identify potential investment risks.

How are Transformer Models used? 
Transformer Models help make this system robust by leveraging their language understanding to extract meaningful insights from the text, such as assessing sentiment and understanding context. This information is vital for determining potential financial risks for investors. Transformer Models are used primarily for sentiment analysis of the financial news articles.
1. Sentiment Analysis:

- The sentiment_analysis function uses a pre-trained transformer model from the HuggingFace Transformers library. Specifically, a transformer-based model such as DistilBERT or BERT is used for analyzing the sentiment of news articles.

- These models have been pre-trained on massive corpora of textual data, which include social media texts, news articles, and other publicly available datasets. They can effectively understand the nuances of human language, making them well-suited for understanding financial news and determining if the overall sentiment is positive or negative.

- Transformer Models help in determining the market's perception of a company, which directly impacts the financial risk. For instance, if negative news is prevalent, it might indicate a higher risk for investors.
2. Language Understanding:
- Transformer Models are also used in performing Named Entity Recognition (NER). Though in the current implementation we use SpaCy's model, larger Transformer Models like BERT or similar models could be used for more advanced entity recognition.
- Transformer Models recognize named entities (e.g., company names, monetary amounts, locations), which are critical to understanding the topics discussed in the news articles. Recognizing entities like "bankruptcy," "lawsuit," or a particular company's name allows the system to assess which pieces of news are more impactful.

Overview and Learning Objectives
- This project aims to teach students how to use Natural Language Processing (NLP) tools to assess financial risks based on textual data.
-  you will learn how to gather data from news articles, perform sentiment analysis, and extract entities to identify financial risk indicators.
- The key technologies being used in this project are:
- HuggingFace Transformers for sentiment analysis, leveraging pre-trained Transformer Models to determine positive or negative sentiment in news articles.
- SpaCy for Named Entity Recognition (NER) to detect key entities like companies, events, and financial terms.
- BeautifulSoup for web scraping to gather recent news articles from Google News.
- Pandas and Matplotlib for data manipulation and visualization.
- The data used includes scraped financial news articles for companies like Tesla, NIO, Rivian, Lucid Motors, QuantumScape, and British American Tobacco which are analyzed to assess their risk potential.


# Step 1: Setting Up the Environment

In [ ]:
# Let's start by installing necessary libraries in Google Colab
# This includes HuggingFace Transformers, SpaCy, and financial sentiment analysis packages.
!pip install transformers
!pip install spacy
!pip install beautifulsoup4
!pip install yfinance

In [ ]:
# Load Spacy's pre-trained model for Named Entity Recognition
!python -m spacy download en_core_web_sm

In [ ]:
# Check if a GPU (CUDA) is available for faster computations; otherwise, use the CPU
# Using GPU is recommended for faster generation as Transformer models can run faster with GPU acceleration.
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"You're using: {gpu_name}")
else:
    print("No GPU detected.")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Importing Libraries and Tools

In [ ]:
import pandas as pd
import numpy as np
import requests
import re
import spacy
from bs4 import BeautifulSoup
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import yfinance as yf
import matplotlib.pyplot as plt

# Step 2: Fetch Financial News Data (Data Collection)


In [ ]:
def get_news(company_name, limit = 5):
  url = f"https://news.google.com/rss/search?q={company_name}+financial+news"
  response = requests.get(url)
  soup = BeautifulSoup(response.text, 'xml')
  headlines = []
  for item in soup.find_all('item'):
    headlines.append(item.title.text)
    if len(headlines) >= limit:
      break
  return headlines

# Example usage to get news articles related to Tesla, NIO, and some smaller companies with acquisition potential

In [ ]:
tesla_news = get_news("Tesla")
nio_news = get_news("NIO")
rivian_news = get_news("Rivian")
lucid_news = get_news("Lucid Motors")
quantumscape_news = get_news("Quantum Scape")
british_american_tobacco_news = get_news("British American Tobacco")

print("Finance news for Tesla:", tesla_news)
print("Finance news for NIO:", nio_news)
print("Finance news for Rivian:", rivian_news)
print("Finance news for Lucid Motors:", lucid_news)
print("Finance news for Quantum Scape:", quantumscape_news)
print("Finance news for British American Tobacco:", british_american_tobacco_news)
# Explanation: This function `get_news` fetches recent financial news headlines for a specific company.
# We use Google News to get relevant articles, which will then be used for sentiment analysis.

# Step 3: Perform Sentiment Analysis (Risk Assessment)



# Details of Pre-trained Models: 
1. Hugging Face Sentiment Analysis Pipeline: This uses Transformer Models like DistilBERT, which is a smaller version of BERT designed to be faster and more efficient, while still capturing important language features.
  - The original BERT (Bidirectional Encoder Representations from Transformers) model is pre-trained on a vast corpus that includes books, Wikipedia, and other publicly available texts. This model is then fine-tuned for specific tasks like sentiment analysis.
- SpaCy's NER: The en_core_web_sm model used here is trained on a variety of English text data, which helps it recognize entities like companies, locations, and events in the financial news.


In [ ]:
def sentiment_analysis(news_list):
  sentiment_analyzer = pipeline("sentiment-analysis", model = "distilbert-base-uncased-finetuned-sst-2-english", device = 0 if torch.cuda.is_available() else -1)
  results = []

  #loop
  for article in news_list:
    analysis = sentiment_analyzer(article)[0]
    results.append({
        'article': article,
        'label': analysis['label'], #positive or negative
        'score': analysis['score'] #confidence score of the sentiment label
    })
  return results

Explanation: The `sentiment_analysis` function takes a list of news articles, and uses HuggingFace's pre-trained sentiment analysis model to determine if the sentiment is positive or negative.
- The output is a list of sentiment scores that indicate the overall sentiment regarding the company.
- This helps us understand if the news generally indicates positive or negative outlook for the company, impacting its financial risk.

In [ ]:
sentiment_results_tesla = sentiment_analysis(tesla_news)
sentiment_results_nio = sentiment_analysis(nio_news)
sentiment_results_rivian = sentiment_analysis(rivian_news)
sentiment_results_lucid = sentiment_analysis(lucid_news)
sentiment_results_quantumscape = sentiment_analysis(quantumscape_news)
sentiment_results_british_american_tobacco = sentiment_analysis(british_american_tobacco_news)

In [ ]:
# Convert results to DataFrames for easier viewing
sentiment_df = pd.concat([
    pd.DataFrame(sentiment_results_tesla),
    pd.DataFrame(sentiment_results_nio),
    pd.DataFrame(sentiment_results_rivian),
    pd.DataFrame(sentiment_results_lucid),
    pd.DataFrame(sentiment_results_quantumscape),
    pd.DataFrame(sentiment_results_british_american_tobacco)
])
sentiment_df

# Step 4: Named Entity Recognition (NER) for Key Entities

In [ ]:
def perform_ner(news_list):
  nlp = spacy.load('en_core_web_sm')
  entities = []

  for article in news_list:
    doc = nlp(article) #doc is a container

  #loop:
    for ent in doc.ents:
      entities.append({
          'article': article,
          'entity.text': ent.text,
          'entity.type': ent.label_ #the type of entity 'ORG' 'GPE'
    })

  return entities


Explanation: Named Entity Recognition (NER) is applied to detect key entities in news articles, such as company names, locations, currency values, and events. This helps identify specific topics being discussed in the news.
SpaCy's pre-trained 'en_core_web_sm' model is used to extract entities.

In [ ]:
ner_results_tesla = perform_ner(tesla_news)
ner_results_nio = perform_ner(nio_news)
ner_results_lucid = perform_ner(lucid_news)
ner_results_rivian = perform_ner(rivian_news)
ner_results_quantumscape = perform_ner(quantumscape_news)
ner_results_british_american_tobacco = perform_ner(british_american_tobacco_news)

In [ ]:
ner_results = pd.concat([
    pd.DataFrame(ner_results_tesla),
    pd.DataFrame(ner_results_nio),
    pd.DataFrame(ner_results_rivian),
    pd.DataFrame(ner_results_lucid),
    pd.DataFrame(ner_results_quantumscape),
    pd.DataFrame(ner_results_british_american_tobacco)
])
ner_results

## Step 5: Risk Assessment Based on Data Analysis

In [ ]:
def assess_risk(sentiment_results, ner_results):
  risk_score = 0

  for result in sentiment_results:
    if result['label'] == 'NEGATIVE':
      risk_score += 1 #negative news increases risk
    elif result['label'] == 'POSITIVE':
      risk_score -= 0.5  #positive news reduces risk

  risk_keywords = ['crisis', 'loss', 'bankruptcy', 'lawsuit', 'decline']
  for entity in ner_results:
    if entity['entity.text'].lower() in risk_keywords:
      risk_score += 2 #high impact on risk

  return risk_score

Explanation of the code above:   The `assess_risk` function combines both sentiment results and entity recognition data to come up with a financial risk score.
-  Sentiment analysis is used to weigh negative or positive news.
- A higher risk score indicates a greater financial risk associated with the company.

In [ ]:
risk_tesla = assess_risk(sentiment_results_tesla, ner_results_tesla)
risk_nio = assess_risk(sentiment_results_nio, ner_results_nio)
risk_rivian = assess_risk(sentiment_results_rivian, ner_results_rivian)
risk_lucid = assess_risk(sentiment_results_lucid, ner_results_lucid)
risk_quantumscape = assess_risk(sentiment_results_quantumscape, ner_results_quantumscape)
risk_british_american_tobacco = assess_risk(sentiment_results_british_american_tobacco, ner_results_british_american_tobacco)

In [ ]:
print(f"Overall Financial Risk Score for Tesla: {risk_tesla}")
print(f"Overall Financial Risk Score for NIO: {risk_nio}")
print(f"Overall Financial Risk Score for Rivian: {risk_rivian}")
print(f"Overall Financial Risk Score for Lucid Motors: {risk_lucid}")
print(f"Overall Financial Risk Score for Quantum Scape: {risk_quantumscape}")
print(f"Overall Financial Risk Score for British American Tobacco: {risk_british_american_tobacco}")

## Step 6: Visualizing the Risk

In [ ]:
#Plotting risk scores for multiple companies
companies = ['Tesla (TSLA)', 'NIO (NIO)', 'Rivian (RIVN)', 'Lucid Motors (LCID)', 'QuantumScape (QS)', 'British American Tobacco (BATS.L)']
risk_scores = [risk_tesla, risk_nio, risk_rivian, risk_lucid, risk_quantumscape, risk_british_american_tobacco]
plt.bar(companies, risk_scores, color = ['#CC3333' if score > 0 else '#3CB371' for score in risk_scores])
plt.title('Financial Risk Assessment')
plt.xlabel('Company')
plt.ylabel('Risk Score')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Import Seaborn for improved visualization
import seaborn as sns

# Set a modern and visually appealing style
plt.style.use('ggplot')  # Use a universally available style

# Plotting risk scores for multiple companies
companies = ['Tesla (TSLA)', 'NIO (NIO)', 'Rivian (RIVN)', 'Lucid Motors (LCID)', 'QuantumScape (QS)', 'British American Tobacco (BATS.L)']
risk_scores = [risk_tesla, risk_nio, risk_rivian, risk_lucid, risk_quantumscape, risk_british_american_tobacco]

# Set the figure size to make it suitable for presentations
plt.figure(figsize=(16, 10))

# Create the bar chart with modern color settings
# Use a gradient of colors to highlight increasing risk, with high-risk being darkest red
bars = plt.bar(companies, risk_scores, color=['#3CB371' if score <= 0 else '#4682B4' if score <= 1 else '#FFBF00' if score <= 2 else '#CC3333' for score in risk_scores], edgecolor='black')
#market blue, risk amber, persian red
# Adding title and labels with enhanced fonts
plt.title('Financial Risk Assessment for Companies', fontsize=24, fontweight='bold', color='#333333', family='DejaVu Sans')
plt.xlabel('Company', fontsize=18, fontweight='bold', family='DejaVu Sans')
plt.ylabel('Risk Score', fontsize=18, fontweight='bold', family='DejaVu Sans')
plt.xticks(rotation=45, fontsize=14, family='DejaVu Sans')
plt.yticks(fontsize=14, family='DejaVu Sans')

# Annotate each bar with the risk score value for better insight
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 0.1, f'{yval:.2f}', ha='center', va='bottom', fontsize=14, color='black', fontweight='bold', family='DejaVu Sans')

# Add annotations to highlight significant risks, including emoji to make it more engaging
high_risk_threshold = 2  # Define a threshold for high risk
for i, score in enumerate(risk_scores):
    if score > high_risk_threshold:
        plt.annotate(
            '⚠️ High Risk',
            xy=(i, score),
            xytext=(i, score + 1),
            textcoords='offset points',
            arrowprops=dict(facecolor='red', arrowstyle='->', lw=2),
            fontsize=12,
            fontweight='bold',
            color='red',
            family='DejaVu Sans'
        )

# Add gridlines to improve readability
plt.grid(axis='y', linestyle='--', linewidth=0.7)

# Tighten layout for a polished look
plt.tight_layout()

# Display the updated plot
plt.show()

Explanation: The final bar chart visualizes the financial risk score for each company. Lower or negative scores indicate more favorable sentiment, while higher scores indicate greater sentiment-driven risk. The color scale highlights increasing levels of risk from low to high.

## Interpretation

- Higher risk score → more negative sentiment in news
- Lower risk score → more positive sentiment
- Near zero → mixed or neutral perception

This reflects market sentiment, not actual financial fundamentals.